In [2]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image


In [27]:
PATCH_SIZE = 224
BATCH_SIZE = 32
SEED = 42

PATCHES_ROOT = "patches_dataset/patches_v3_seed42_pad1.6_iouth0.09"
traincsv_path = os.path.join(PATCHES_ROOT, "metadata.csv")
test_csv = "patches_dataset/test_patches_v3/test_metadata.csv"

set_seed(SEED)


In [29]:
AUG_ROOT = "patches_dataset/patches_v3_train_aug"
PATH_AUG_METADATA = os.path.join(AUG_ROOT, "train_aug_metadata.csv")

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt

from PatchDataset import load_dataset, load_augmented_dataset

import pipeline
from pipeline import fit_binary_classifier, predict_probs, set_seed


In [22]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

device: mps


In [42]:
# loading the dataset
traindf = pd.read_csv(traincsv_path)
traindf["patch_id"] = traindf["patch_id"].astype(int)
traindf["label"] = traindf["label"].astype(int)
traindf["type"] = traindf["type"].astype(str)

testdf = pd.read_csv(test_csv)
testdf["patch_id"] = testdf["patch_id"].astype(int)
testdf["label"] = testdf["label"].astype(int)
testdf["type"] = testdf["type"].astype(str)

train_dataset,val_dataset,train_loader,val_loader,test_dataset,test_loader = load_dataset(traindf,testdf,PATCHES_ROOT,BATCH_SIZE)

for xb,yb in train_loader:
    print(xb.shape,yb.shape,min(yb),max(yb))
    break

train/val: 15858 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0) tensor(1)


In [43]:
print("train:", traindf.shape, "test:", testdf.shape)
print(traindf["label"].value_counts(dropna=False).sort_index())
print(testdf["label"].value_counts(dropna=False).sort_index())

train: (19823, 6) test: (3965, 6)
label
0     6246
1    13577
Name: count, dtype: int64
label
0    1249
1    2716
Name: count, dtype: int64


In [11]:
class SimpleCNN3(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1) # 3 bc RGB 
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # archi 16/32/64
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.head = nn.Linear(64, 2) # couche classif finale FC, classif binaire donc 2 scores pour pipeline.py

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x) # pour non linéarité sans vanishing gradient
        x = self.pool(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = F.relu(x)

        x = x.mean(dim=[2, 3])  # Global Average Pooling
        logits = self.head(x)
        return logits


In [44]:
#-----------------------------------------------------------------------------------

In [12]:
def build_model():
    model = SimpleCNN3().to(device)
    return model


def build_loss_and_optimizer(model, train_df, lr=1e-3):
    class_counts = train_df["label"].value_counts().sort_index()
    neg = int(class_counts.get(0, 0))
    pos = int(class_counts.get(1, 0))

    # pondérer la classe positive pour garder rééquilibrage
    # poids classe 0 = 1.0 ; poids classe 1 = neg / pos
    class_weights = torch.tensor([1.0, neg / max(pos, 1)], dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights) # classif binaire avec 2 logits (to use pipeline.ipynb)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    return criterion, optimizer


In [13]:
# helper pour lancer une expérience pipeline.py sur un jeu de loaders
def run_experiment(train_dataset, val_dataset, train_loader, val_loader, test_dataset, test_loader, run_dir, train_df, experiment_name):
    model = build_model()
    criterion, optimizer = build_loss_and_optimizer(model, train_df, lr=1e-3)

    config = {
        "model": "SimpleCNN3",
        "batch_size": BATCH_SIZE,
        "epochs": 10,
        "learning_rate": 1e-3,
        "loss": "CrossEntropyLoss",
        "patch_size": PATCH_SIZE,
        "seed": SEED,
        "experiment": experiment_name,
    }

    pipeline.SAVING_FILE_NAME = os.path.join(run_dir, "best_model")

    run_info, best_model = fit_binary_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        run_dir=run_dir,
        config=config,
        patience=5,
        monitor="f1",
    )

    y_true, scores = predict_probs(best_model, test_loader, device)
    preds05 = (scores >= 0.5).astype(int)
    preds07 = (scores >= 0.7).astype(int)

    print(experiment_name)
    print("test @0.5")
    print(confusion_matrix(y_true, preds05))
    print(classification_report(y_true, preds05, target_names=["Negative", "Positive"]))

    return {
        "run_info": run_info,
        "best_model": best_model,
        "y_true": y_true,
        "scores": scores,
        "preds05": preds05,
        "preds07": preds07,
    }


In [14]:
# loading the dataset
train_dataset, val_dataset, train_loader, val_loader, test_dataset, test_loader = load_dataset(
    traindf, testdf, PATCHES_ROOT, BATCH_SIZE
)

for xb, yb in train_loader:
    print(xb.shape, yb.shape, min(yb), max(yb))
    break


train/val: 15858 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0) tensor(1)


In [15]:
normal_results = run_experiment(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    train_loader=train_loader,
    val_loader=val_loader,
    test_dataset=test_dataset,
    test_loader=test_loader,
    run_dir="runs/cnn_boxes_v3_normal",
    train_df=traindf,
    experiment_name="CNN on v3 normal",
)


Model improved
Epoch 001 | train loss 0.5540 f1 0.7848 acc 0.7231 | val loss 0.5136 f1 0.7846 acc 0.7291 | 
Model improved
Epoch 002 | train loss 0.5094 f1 0.8143 acc 0.7584 | val loss 0.4706 f1 0.8384 acc 0.7849 | 
patience 1/5
Epoch 003 | train loss 0.4848 f1 0.8212 acc 0.7674 | val loss 0.4695 f1 0.8362 acc 0.7828 | 
patience 2/5
Epoch 004 | train loss 0.4661 f1 0.8277 acc 0.7753 | val loss 0.4805 f1 0.7790 acc 0.7314 | 
patience 3/5
Epoch 005 | train loss 0.4615 f1 0.8313 acc 0.7792 | val loss 0.4520 f1 0.7975 acc 0.7485 | 
patience 4/5
Epoch 006 | train loss 0.4452 f1 0.8413 acc 0.7917 | val loss 0.4270 f1 0.8381 acc 0.7897 | 
Model improved
Epoch 007 | train loss 0.4364 f1 0.8434 acc 0.7940 | val loss 0.4044 f1 0.8592 acc 0.8149 | 
patience 1/5
Epoch 008 | train loss 0.4230 f1 0.8529 acc 0.8047 | val loss 0.4375 f1 0.8011 acc 0.7571 | 
Model improved
Epoch 009 | train loss 0.4246 f1 0.8520 acc 0.8038 | val loss 0.4022 f1 0.8771 acc 0.8320 | 
patience 1/5
Epoch 010 | train loss 0.

In [ ]:
# loading the augmented dataset
aug_train_dataset,aug_val_dataset,aug_train_loader,aug_val_loader,aug_test_dataset,aug_test_loader = load_augmented_dataset(
    PATCHES_ROOT,
    AUG_ROOT,
    PATH_AUG_METADATA,
    test_csv,
    BATCH_SIZE
)

for xb, yb in aug_train_loader:
    print(xb.shape, yb.shape, min(yb), max(yb))
    break


train_aug/test: 31716 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0) tensor(1)


In [39]:
augmented_results = run_experiment(
    train_dataset=aug_train_dataset,
    val_dataset=aug_val_dataset,
    train_loader=aug_train_loader,
    val_loader=aug_val_loader,
    test_dataset=aug_test_dataset,
    test_loader=aug_test_loader,
    run_dir="runs/cnn_boxes_v3_augmented",
    train_df=traindf,
    experiment_name="CNN on v3 augmented",
)


NameError: name 'aug_val_dataset' is not defined

In [ ]:
run_dir = "runs/cnn_boxes_v3_augmented"

model = SimpleCNN3().to(device)

train_labels = [int(aug_train_dataset[i][1]) for i in range(len(aug_train_dataset))]
pos = sum(train_labels)
neg = len(train_labels) - pos

class_weights = torch.tensor(
    [1.0, neg / max(pos, 1)],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = None

config = {
    "model": "SimpleCNN3",
    "epochs": 10,
    "batch_size": BATCH_SIZE,
    "lr": 1e-3,
    "loss": "CrossEntropyLoss",
    "dataset": "patches_v3_augmented",
}

pipeline.SAVING_FILE_NAME = os.path.join(run_dir, "best_model")

run_info, best_model = fit_binary_classifier(
    model=model,
    train_loader=aug_train_loader,
    val_loader=aug_val_loader,
    test_loader=aug_test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    run_dir=run_dir,
    config=config,
    scheduler=scheduler,
)

In [ ]:
# pr quels résultats afficher
results = normal_results
y_true = results["y_true"]
scores = results["scores"]

# y score probas so we make the preds by thresholding at 0.5
preds = (scores >= 0.5).astype(int)

y_true, scores[:10], preds[:10]


In [ ]:
fpr, tpr, _ = roc_curve(y_true, scores)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC AUC={roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC")
plt.legend()
plt.show()


In [ ]:
prec, rec, _ = precision_recall_curve(y_true, scores)
ap = average_precision_score(y_true, scores)

plt.figure()
plt.plot(rec, prec, label=f"AP (PR AUC)={ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall")
plt.legend()
plt.show()


In [ ]:
print(confusion_matrix(y_true, preds))

# { TN FP 
# FN TP }


In [ ]:
print("                            Métriques")
print(classification_report(y_true, preds, target_names=["Negative", "Positive"]))


In [ ]:
preds = (scores >= 0.7).astype(int) # moins bon, seuil plus serré...
print(confusion_matrix(y_true, preds))

# { TN FP 
# FN TP }


In [ ]:
print("                            Métriques")
print(classification_report(y_true, preds, target_names=["Negative", "Positive"]))


In [ ]:
# oversampling des negatifs ? pour equlibrer dataset
# taille kernel ?
# reduire learning rate 
# automatiser test avec differents seeds
# essayer de ne pas manquer d'arbres, trop de faux negatifs cf precision negative.
